# MiniGPT-Stack pretraining launcher (THIN — all logic lives in train/)

Run this notebook **under Save Version (GPU ON, internet ON)** so /kaggle/working persists as a version output.
Flow: env check -> clone/pull -> install (keep preinstalled torch) -> restore last checkpoint ->
shards -> shakedown/sweep (first session) or main run -> HF backup. Then Save Version.

In [ ]:
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "enable GPU in notebook settings"
# Record these versions in docs/decisions.md with the run results.

In [ ]:
import os
REPO = "https://github.com/Priyanshaur/Glassbox.git"
ROOT = "/kaggle/working/Glassbox"
if not os.path.exists(ROOT):
    !git clone $REPO $ROOT
else:
    !git -C $ROOT pull --ff-only
%cd $ROOT
!git rev-parse --short HEAD

In [ ]:
# Install everything EXCEPT torch: Kaggle's preinstalled torch stays (see decisions.md).
!grep -v '^torch' requirements.txt | pip install -r /dev/stdin
!python -m pytest tests/test_config.py tests/test_model.py -q 2>&1 | tail -n 3

## Restore previous checkpoints
Attach the previous version's output as a Kaggle input dataset first (Add Data -> previous run output).
The cell below finds the newest ckpt_step*.pt under any attached input and copies the whole
checkpoint dir into working, so `--resume latest` continues the loss curve.

In [ ]:
import glob, os, shutil
cands = sorted(glob.glob("/kaggle/input/*/checkpoints/*/*.pt")) +
        sorted(glob.glob("/kaggle/input/*/ckpt_step*.pt"))
if cands:
    src_dir = os.path.dirname(cands[-1])
    dst_dir = os.path.join(os.getcwd(), os.path.relpath(src_dir, "/kaggle/input")) \
        if src_dir.startswith("/kaggle/input") else os.path.join("checkpoints", "restored")
    # Simpler robust rule: mirror into ./checkpoints/<same-run-name>/
    dst_dir = os.path.join("checkpoints", os.path.basename(src_dir))
    os.makedirs(dst_dir, exist_ok=True)
    for f in glob.glob(os.path.join(src_dir, "*")):
        if os.path.basename(f) not in os.listdir(dst_dir):
            shutil.copy(f, dst_dir)
    print("restored", len(os.listdir(dst_dir)), "files ->", dst_dir)
    print("newest:", max(glob.glob(os.path.join(dst_dir, 'ckpt_step*.pt'))))
else:
    print("no previous checkpoints attached; starting fresh")

## Shards
Attach the tinystories-shards dataset if it exists. Otherwise run the FULL prepare once
(streams ~2GB, ~10-20 min) and Save Version so later sessions attach it instead of redoing it.

In [ ]:
import glob
hit = glob.glob("/kaggle/input/*tinystories*/data/meta.json") + glob.glob("/kaggle/input/*/meta.json")
if hit:
    !ln -sfn $(dirname $hit[0]) data 2>/dev/null || echo "attach manually: use $hit[0] as --out_dir parent"
    print("shards attached:", hit[0])
else:
    print("no shards attached: running FULL prepare (save version afterwards!)")
    # !python -m data.prepare --tokenizer tokenizer/tok8192.json --out_dir data  # full run, no --max_docs

## Shakedown (first GPU session): throughput probe + fp16 stability + kill-resume drill
Probe micro_batch_seqs 8/16/32 for ~60 steps each (fresh tmp dirs); read tok/s from the log lines.
Watch `skips=`: fp16 overflow gate is skips/step < ~1%. Then the drill: 150 steps, clean stop,
resume to 300, and confirm metrics.jsonl continues without a loss jump.

In [ ]:
# Throughput probe (uncomment to run; ~5 min total)
# for MB in 8 16 32:
#     !python -m train.pretrain --config configs/base_30m.yaml --resume none \
#         --micro_batch_seqs $MB --max_steps 60 --ckpt_dir /tmp/probe_$MB 2>&1 | grep pretrain

In [ ]:
# Kill-and-resume drill on GPU (uncomment; uses step-saving, proves continuity)
# !python -m train.pretrain --config configs/base_30m.yaml --resume none --max_steps 150 2>&1 | tail -n 5
# !python -m train.pretrain --config configs/base_30m.yaml --resume latest --max_steps 300 2>&1 | tail -n 5
# !tail -n 4 checkpoints/base_30m/metrics.jsonl

## LR sweep: 3 values x ~900 steps, same schedule, compare val loss at the same step.
Prefer the LOWER lr if within noise. Provisional default 6e-4.

In [ ]:
# LR sweep (uncomment; ~3 x 15 min at 65k tok/step and ~80k tok/s)
# for LR in 3e-4 6e-4 1e-3:
#     !python -m train.pretrain --config configs/base_30m.yaml --resume none \
#         --lr $LR --max_steps 900 --ckpt_dir /tmp/sweep_$LR 2>&1 | tail -n 2
#     !grep val /tmp/sweep_$LR/metrics.jsonl | tail -n 1

## Main run: resume latest, clean-exit before the ~9h kill. Then Save Version.

In [ ]:
!python -m train.pretrain --config configs/base_30m.yaml --resume latest --max_hours 8

## Backup: newest checkpoint -> private Hugging Face repo (skips gracefully).
Put a fine-grained token in Kaggle Secrets as HF_TOKEN. Weights-only copies
(weights_step{N}.pt, saved automatically by train/checkpoint.py) are what later phases load.

In [ ]:
import glob, os
from kaggle_secrets import UserSecretsClient
try:
    tok = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    tok = None
    print("no HF_TOKEN secret; skipping backup:", e)
ckpts = sorted(glob.glob("checkpoints/*/ckpt_step*.pt"))
if tok and ckpts:
    try:
        import importlib
        try:
            hhub = importlib.import_module("huggingface_hub")
        except ImportError:
            !pip install -q huggingface_hub
            hhub = importlib.import_module("huggingface_hub")
        hhub.login(token=tok)
        api = hhub.HfApi()
        repo = api.create_repo("minigpt-stack-ckpts", private=True, exist_ok=True).repo_id
        for f in ckpts[-2:]:  # newest two only; weights-only rides along on Save Version
            api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=repo)
        print("backed up to", repo, ckpts[-2:])
    except Exception as e:
        print("backup skipped (will retry next session):", e)
elif not ckpts:
    print("no checkpoints to back up")